# 20 · Nonlinear fiber and digital backpropagation

Chromatic dispersion ([19](19-chromatic-dispersion.ipynb)) is linear: it does not care how much
power the signal carries.  The **Kerr effect** does: the refractive index
of silica grows with the optical intensity, so a strong signal modulates its
own phase (self-phase modulation) and, with two polarizations, that of the
other (cross-polarization modulation), while dispersion keeps reshaping the
waveform.  The combination is described by the nonlinear Schrödinger
equation (the Manakov equation for two polarizations).

A coherent link over amplified spans of standard fiber therefore has an
optimum launch power.  Below it, amplifier noise (ASE) dominates and the SNR grows 1 dB per
dB of power; above it, the Kerr effect distorts the signal and the SNR
falls, about 2 dB per dB.  Electronic dispersion compensation (CDC) undoes
only the linear part; digital backpropagation (DBP) runs the fiber model
backward and also undoes the deterministic Kerr distortion, which moves the
optimum to a higher power and a higher SNR.

`apply_fiber_propagation` is the channel (split-step Fourier, Manakov
equation for two polarizations, an amplifier after every span) and
`correct_fiber_propagation` is DBP.  Both take the same `Fiber` and link
arguments.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.impairments import Fiber

# Build the data on the GPU when there is one; every later step follows it.
DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()

## 1. Link

Dual-polarization 16-QAM at 32 GBd and 2 samples per symbol (64 GS/s),
with a root-raised-cosine pulse of roll-off 0.1, over
8 spans of 80 km of standard fiber (0.2 dB/km, 17 ps/(nm km),
γ = 1.3 /(W km)), each followed by an amplifier with a 5 dB noise figure.
`Fiber()` holds these defaults.

In [ ]:
C16 = Constellation.qam(16)
tx = ck.generate(
    C16,
    num_symbols=2**14,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    num_channels=2,
    rng=1,
    device=DEVICE,
)
# An optical field on a 193.4 THz (1550.1 nm) carrier: the fiber functions
# take the wavelength from it.
tx = tx.replace(domain="OPT", center_frequency=193.4e12)
ref = tx.reference.symbols
xp = ck.backend.get_array_module(tx.samples)

fiber = Fiber()
LINK = dict(fiber=fiber, span_length_km=80.0, num_spans=8)

## 2. Receiver

The fiber model works in the input's units: the transmitted power stands for
the launch power, so the received field comes back at the transmitter's
scale.  After compensation, the matched filter and the symbol-rate
decimation give one sample per symbol.  The Kerr effect also rotates the
whole constellation by a common phase, which a carrier recovery would remove;
here it is removed against the reference before the SNR is measured.

In [ ]:
def snr_db(field):
    y = ck.filtering.matched_filter(field)
    y = ck.multirate.decimate_to_symbol_rate(y).samples
    common = xp.sum(y * xp.conj(ref), axis=-1, keepdims=True)
    y = y * xp.conj(common) / xp.abs(common)
    return float(np.mean(ck.metrics.snr(y, ref)))


def cdc(rx):
    return ck.filtering.correct_chromatic_dispersion(
        rx,
        dispersion_ps_nm_km=fiber.dispersion_ps_nm_km,
        fiber_length_km=LINK["span_length_km"] * LINK["num_spans"],
    )

## 3. Launch-power sweep

Each power runs the channel with 20 steps per span and compensates the
received field three ways: CDC, DBP with one step per span and `ξ = 0.5`
(the low-complexity setting, which needs `nonlinear_scale` below 1), and DBP
with the channel's own 20 steps (the exact inverse of the deterministic
fiber; it cannot undo the interaction of the signal with the ASE added along
the link).

In [ ]:
powers = np.arange(-2.0, 15.0, 2.0)  # dBm, both polarizations
snr = {"CDC": [], "DBP 1 step/span, ξ = 0.5": [], "DBP 20 steps/span": []}
for p in powers:
    rx = ck.impairments.apply_fiber_propagation(
        tx, launch_power_dbm=p, steps_per_span=20, noise_figure_db=5.0, rng=2, **LINK
    )
    snr["CDC"].append(snr_db(cdc(rx)))
    snr["DBP 1 step/span, ξ = 0.5"].append(
        snr_db(
            ck.filtering.correct_fiber_propagation(
                rx, launch_power_dbm=p, steps_per_span=1, nonlinear_scale=0.5, **LINK
            )
        )
    )
    snr["DBP 20 steps/span"].append(
        snr_db(
            ck.filtering.correct_fiber_propagation(
                rx, launch_power_dbm=p, steps_per_span=20, **LINK
            )
        )
    )

for name, values in snr.items():
    best = int(np.argmax(values))
    print(f"{name:26s} best {values[best]:5.1f} dB at {powers[best]:+.0f} dBm")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, values in snr.items():
    ax.plot(powers, values, marker="o", label=name)
ax.set_xlabel("Launch power (dBm)")
ax.set_ylabel("SNR (dB)")
ax.set_title("8 x 80 km, 32 GBd DP-16QAM")
ax.legend()
plt.show()

## 4. What else the model covers

- `Fiber(pmd_ps_sqrt_km=...)` adds polarization-mode dispersion (random
  waveplates, one per step, drawn from `rng`); DBP leaves it to the adaptive
  MIMO equalizer, as a receiver must.
- `Fiber(dispersion_slope_ps_nm2_km=0.057)` adds third-order dispersion,
  which matters for wideband (multi-channel) fields.
- `steps_per_span` sets the accuracy of the channel: the split-step error
  falls as the square of the step length.

**Next:** [21 · FEC in optical standards](21-fec-in-optical-standards.ipynb).